In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy pandas
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Algoritma Shor

*Estimasi penggunaan: Tiga detik pada prosesor Eagle r3 (CATATAN: Ini hanya perkiraan. Waktu eksekusi kamu bisa berbeda.)*

[Algoritma Shor,](https://epubs.siam.org/doi/abs/10.1137/S0036144598347011) yang dikembangkan oleh Peter Shor pada tahun 1994, adalah algoritma kuantum revolusioner untuk memfaktorkan bilangan bulat dalam waktu polinomial. Signifikansinya terletak pada kemampuannya memfaktorkan bilangan bulat besar secara eksponensial lebih cepat dari algoritma klasik mana pun yang diketahui, mengancam keamanan sistem kriptografi yang banyak digunakan seperti RSA, yang bergantung pada sulitnya memfaktorkan bilangan besar. Dengan menyelesaikan masalah ini secara efisien pada komputer kuantum yang cukup powerful, algoritma Shor berpotensi merevolusi bidang-bidang seperti kriptografi, keamanan siber, dan matematika komputasi — menegaskan kekuatan transformatif dari komputasi kuantum.

Tutorial ini berfokus pada demonstrasi algoritma Shor dengan memfaktorkan 15 di komputer kuantum.

Pertama, kita mendefinisikan masalah pencarian orde dan membangun Circuit yang sesuai dari protokol estimasi fase kuantum. Selanjutnya, kita menjalankan Circuit pencarian orde di hardware nyata menggunakan Circuit dengan kedalaman terpendek yang bisa kita transpilasikan. Bagian terakhir melengkapi algoritma Shor dengan menghubungkan masalah pencarian orde ke faktorisasi bilangan bulat.

Kita mengakhiri tutorial ini dengan diskusi tentang demonstrasi lain dari algoritma Shor di hardware nyata, berfokus pada implementasi generik maupun yang disesuaikan untuk memfaktorkan bilangan bulat tertentu seperti 15 dan 21.
Catatan: Tutorial ini lebih berfokus pada implementasi dan demonstrasi Circuit yang berkaitan dengan algoritma Shor. Untuk sumber edukasi yang lebih mendalam tentang materi ini, silakan lihat kursus [Fundamentals of quantum algorithms](/learning/courses/fundamentals-of-quantum-algorithms/phase-estimation-and-factoring/introduction) oleh Dr. John Watrous, dan makalah-makalah di bagian [Referensi](#references).
### Persyaratan
Sebelum memulai tutorial ini, pastikan kamu sudah menginstal:
- Qiskit SDK v2.0 atau lebih baru, dengan dukungan [visualisasi](https://docs.quantum.ibm.com/api/qiskit/visualization)
- Qiskit Runtime v0.40 atau lebih baru (`pip install qiskit-ibm-runtime`)
### Persiapan

In [ ]:
import numpy as np
import pandas as pd
from fractions import Fraction
from math import floor, gcd, log

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import QFT, UnitaryGate
from qiskit.transpiler import CouplingMap, generate_preset_pass_manager
from qiskit.visualization import plot_histogram

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler

## Langkah 1: Memetakan input klasik ke masalah kuantum
### Latar belakang
Algoritma Shor untuk faktorisasi bilangan bulat menggunakan masalah perantara yang dikenal sebagai masalah *pencarian orde*. Di bagian ini, kita mendemonstrasikan cara menyelesaikan masalah pencarian orde menggunakan *estimasi fase kuantum*.
### Masalah estimasi fase
Dalam masalah estimasi fase, kita diberikan keadaan kuantum $\ket{\psi}$ dari $n$ Qubit, bersama dengan Circuit kuantum yang bekerja pada $n$ Qubit. Kita dijanjikan bahwa $\ket{\psi}$ adalah vektor eigen dari matriks uniter $U$ yang menggambarkan aksi Circuit tersebut, dan tujuan kita adalah menghitung atau mendekati nilai eigen $\lambda = e^{2 \pi i \theta}$ yang berkorespondensi dengan $\ket{\psi}$. Dengan kata lain, Circuit harus menghasilkan aproksimasi dari bilangan $\theta \in [0, 1)$ yang memenuhi $$U \ket{\psi}= e^{2 \pi i \theta} \ket{\psi}.$$
Tujuan dari Circuit estimasi fase adalah untuk mengaproksimasi $\theta$ dalam $m$ bit. Secara matematis, kita ingin menemukan $y$ sedemikian sehingga $\theta \approx y / 2^m$, di mana $y \in {0, 1, 2, \dots, 2^{m-1}}$. Gambar berikut menunjukkan Circuit kuantum yang mengestimasi $y$ dalam $m$ bit dengan melakukan pengukuran pada $m$ Qubit.
![Quantum phase estimation circuit](../learning/images/courses/fundamentals-of-quantum-algorithms/phase-estimation-and-factoring/phase-estimation-procedure.svg)
Pada Circuit di atas, $m$ Qubit teratas diinisialisasi dalam keadaan $\ket{0^m}$, dan $n$ Qubit terbawah diinisialisasi dalam $\ket{\psi}$, yang dijanjikan sebagai vektor eigen dari $U$. Komponen pertama dalam Circuit estimasi fase adalah operasi uniter terkontrol yang bertanggung jawab melakukan *phase kickback* ke Qubit kontrol yang bersesuaian. Uniter-uniter terkontrol ini dipangkatkan sesuai posisi Qubit kontrol, mulai dari bit paling tidak signifikan hingga bit paling signifikan. Karena $\ket{\psi}$ adalah vektor eigen dari $U$, keadaan $n$ Qubit bawah tidak terpengaruh oleh operasi ini, tetapi informasi fase dari nilai eigen merambat ke $m$ Qubit teratas.
Ternyata, setelah operasi phase kickback melalui uniter-uniter terkontrol, semua keadaan yang mungkin dari $m$ Qubit teratas saling ortogonal untuk setiap vektor eigen $\ket{\psi}$ dari uniter $U$. Oleh karena itu, keadaan-keadaan ini dapat dibedakan dengan sempurna, dan kita dapat memutar basis yang mereka bentuk kembali ke basis komputasi untuk melakukan pengukuran. Analisis matematis menunjukkan bahwa matriks rotasi ini berkorespondensi dengan transformasi Fourier kuantum invers (QFT) dalam ruang Hilbert berdimensi $2^m$. Intuisi di balik ini adalah bahwa struktur periodik dari operator eksponensiasi modular dikodekan dalam keadaan kuantum, dan QFT mengubah periodisitas ini menjadi puncak yang terukur dalam domain frekuensi.

Untuk pemahaman yang lebih mendalam tentang mengapa Circuit QFT digunakan dalam algoritma Shor, kita merujuk pembaca ke kursus [Fundamentals of quantum algorithms](/learning/courses/fundamentals-of-quantum-algorithms/phase-estimation-and-factoring/introduction).
Kita sekarang siap menggunakan Circuit estimasi fase untuk pencarian orde.
### Masalah pencarian orde
Untuk mendefinisikan masalah pencarian orde, kita mulai dengan beberapa konsep teori bilangan. Pertama, untuk setiap bilangan bulat positif $N$, definisikan himpunan $\mathbb{Z}_N$ sebagai $$\mathbb{Z}_N = {0, 1, 2, \dots, N-1}.$$
Semua operasi aritmetika di $\mathbb{Z}_N$ dilakukan modulo $N$. Khususnya, semua elemen $a \in \mathbb{Z}_n$ yang koprima dengan $N$ bersifat istimewa dan membentuk $\mathbb{Z}^*_N$ sebagai $$\mathbb{Z}^*_N = { a \in \mathbb{Z}_N : \mathrm{gcd}(a, N)=1 }.$$
Untuk suatu elemen $a \in \mathbb{Z}^*_N$, bilangan bulat positif terkecil $r$ sedemikian sehingga $$a^r \equiv 1 \; (\mathrm{mod} \; N)$$ didefinisikan sebagai *orde* dari $a$ modulo $N$. Seperti yang akan kita lihat nanti, menemukan orde dari $a \in \mathbb{Z}^*_N$ akan memungkinkan kita memfaktorkan $N$.
Untuk membangun Circuit pencarian orde dari Circuit estimasi fase, kita perlu dua pertimbangan. Pertama, kita perlu mendefinisikan uniter $U$ yang akan memungkinkan kita menemukan orde $r$, dan kedua, kita perlu mendefinisikan vektor eigen $\ket{\psi}$ dari $U$ untuk mempersiapkan keadaan awal Circuit estimasi fase.

Untuk menghubungkan masalah pencarian orde ke estimasi fase, kita mempertimbangkan operasi yang didefinisikan pada sistem yang keadaan klasiknya berkorespondensi dengan $\mathbb{Z}_N$, di mana kita mengalikan dengan elemen tetap $a \in \mathbb{Z}^*_N$. Khususnya, kita mendefinisikan operator perkalian $M_a$ sedemikian sehingga $$M_a \ket{x} = \ket{ax \; (\mathrm{mod} \; N)}$$ untuk setiap $x \in \mathbb{Z}_N$. Perlu dicatat bahwa secara implisit kita mengambil hasil perkalian modulo $N$ di dalam ket di ruas kanan persamaan. Analisis matematis menunjukkan bahwa $M_a$ adalah operator uniter. Lebih lanjut, ternyata $M_a$ memiliki pasangan vektor eigen dan nilai eigen yang memungkinkan kita menghubungkan orde $r$ dari $a$ ke masalah estimasi fase. Secara spesifik, untuk pilihan $j \in {0, \dots, r-1}$ mana pun, kita memiliki bahwa $$\ket{\psi_j} = \frac{1}{\sqrt{r}} \sum^{r-1}_{k=0} \omega^{-jk}_{r} \ket{a^k}$$ adalah vektor eigen dari $M_a$ yang nilai eigennya berkorespondensi adalah $\omega^{j}_{r}$, di mana $$\omega^{j}_{r} = e^{2 \pi i \frac{j}{r}}.$$
Dari pengamatan, kita melihat bahwa pasangan vektor eigen/nilai eigen yang nyaman adalah keadaan $\ket{\psi_1}$ dengan $\omega^{1}_{r} = e^{2 \pi i \frac{1}{r}}$. Oleh karena itu, jika kita bisa menemukan vektor eigen $\ket{\psi_1}$, kita bisa mengestimasi fase $\theta=1/r$ dengan Circuit kuantum kita dan dengan demikian mendapatkan estimasi orde $r$. Namun, hal itu tidak mudah dilakukan, dan kita perlu mempertimbangkan alternatif lain.

Mari kita pertimbangkan apa yang akan dihasilkan Circuit jika kita mempersiapkan keadaan komputasi $\ket{1}$ sebagai keadaan awal. Ini bukan keadaan eigen dari $M_a$, tetapi merupakan superposisi seragam dari keadaan-keadaan eigen yang baru saja kita deskripsikan. Dengan kata lain, relasi berikut berlaku. $$ \ket{1} = \frac{1}{\sqrt{r}} \sum^{r-1}_{k=0} \ket{\psi_k} $$
Implikasi dari persamaan di atas adalah bahwa jika kita menetapkan keadaan awal ke $\ket{1}$, kita akan memperoleh hasil pengukuran yang persis sama seolah-olah kita telah memilih $k \in { 0, \dots, r-1}$ secara seragam acak dan menggunakan $\ket{\psi_k}$ sebagai vektor eigen dalam Circuit estimasi fase. Dengan kata lain, pengukuran pada $m$ Qubit teratas menghasilkan aproksimasi $y / 2^m$ dari nilai $k / r$ di mana $k \in { 0, \dots, r-1}$ dipilih secara seragam acak. Ini memungkinkan kita mempelajari $r$ dengan tingkat keyakinan yang tinggi setelah beberapa eksekusi independen, yang merupakan tujuan kita.
### Operator eksponensiasi modular
Sejauh ini, kita menghubungkan masalah estimasi fase ke masalah pencarian orde dengan mendefinisikan $U = M_a$ dan $\ket{\psi} = \ket{1}$ dalam Circuit kuantum kita. Oleh karena itu, bahan terakhir yang tersisa adalah menemukan cara efisien untuk mendefinisikan eksponen modular dari $M_a$ sebagai $M_a^k$ untuk $k = 1, 2, 4, \dots, 2^{m-1}$.
Untuk melakukan komputasi ini, kita menemukan bahwa untuk pangkat $k$ mana pun yang kita pilih, kita dapat membuat Circuit untuk $M_a^k$ bukan dengan mengiterasi $k$ kali Circuit untuk $M_a$, melainkan dengan menghitung $b = a^k \; \mathrm{mod} \; N$ dan kemudian menggunakan Circuit untuk $M_b$. Karena kita hanya membutuhkan pangkat yang merupakan pangkat dua itu sendiri, kita dapat melakukan ini secara efisien secara klasik menggunakan pengkuadratan iteratif.
## Langkah 2: Mengoptimalkan masalah untuk eksekusi pada hardware kuantum
### Contoh spesifik dengan $N = 15$ dan $a=2$
Kita bisa berhenti sejenak di sini untuk mendiskusikan contoh spesifik dan membangun Circuit pencarian orde untuk $N=15$. Perhatikan bahwa kemungkinan $a \in \mathbb{Z}_N^*$ yang non-trivial untuk $N=15$ adalah $a \in {2, 4, 7, 8, 11, 13, 14 }$. Untuk contoh ini, kita pilih $a=2$. Kita akan membangun operator $M_2$ dan operator eksponensiasi modular $M_2^k$.
Aksi dari $M_2$ pada keadaan basis komputasi adalah sebagai berikut.
$$M_2 \ket{0} = \ket{0} \quad M_2 \ket{5} = \ket{10} \quad M_2 \ket{10} = \ket{5}$$
$$M_2 \ket{1} = \ket{2} \quad M_2 \ket{6} = \ket{12} \quad M_2 \ket{11} = \ket{7}$$
$$M_2 \ket{2} = \ket{4} \quad M_2 \ket{7} = \ket{14} \quad M_2 \ket{12} = \ket{9}$$
$$M_2 \ket{3} = \ket{6} \quad M_2 \ket{8} = \ket{1} \quad M_2 \ket{13} = \ket{11}$$
$$M_2 \ket{4} = \ket{8} \quad M_2 \ket{9} = \ket{3} \quad M_2 \ket{14} = \ket{13}$$
Dari pengamatan, kita bisa melihat bahwa keadaan basis diacak, sehingga kita memiliki matriks permutasi. Kita dapat membangun operasi ini pada empat Qubit dengan Gate swap. Di bawah ini, kita membangun operasi $M_2$ dan controlled-$M_2$.

In [2]:
def M2mod15():
    """
    M2 (mod 15)
    """
    b = 2
    U = QuantumCircuit(4)

    U.swap(2, 3)
    U.swap(1, 2)
    U.swap(0, 1)

    U = U.to_gate()
    U.name = f"M_{b}"

    return U

In [3]:
# Get the M2 operator
M2 = M2mod15()

# Add it to a circuit and plot
circ = QuantumCircuit(4)
circ.compose(M2, inplace=True)
circ.decompose(reps=2).draw(output="mpl", fold=-1)

<Image src="../docs/images/tutorials/shors-algorithm/extracted-outputs/0a8885f1-91d4-40bd-912d-dc5eea05f5bd-0.avif" alt="Output of the previous code cell" />

In [4]:
def controlled_M2mod15():
    """
    Controlled M2 (mod 15)
    """
    b = 2
    U = QuantumCircuit(4)

    U.swap(2, 3)
    U.swap(1, 2)
    U.swap(0, 1)

    U = U.to_gate()
    U.name = f"M_{b}"
    c_U = U.control()

    return c_U

In [5]:
# Get the controlled-M2 operator
controlled_M2 = controlled_M2mod15()

# Add it to a circuit and plot
circ = QuantumCircuit(5)
circ.compose(controlled_M2, inplace=True)
circ.decompose(reps=1).draw(output="mpl", fold=-1)

<Image src="../docs/images/tutorials/shors-algorithm/extracted-outputs/ab7fe331-2f9e-47ca-ba3b-f5d67992062a-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/shors-algorithm/extracted-outputs/ab7fe331-2f9e-47ca-ba3b-f5d67992062a-0.avif)

Gate yang bekerja pada lebih dari dua Qubit akan didekomposisi lebih lanjut menjadi Gate dua Qubit.

In [6]:
circ.decompose(reps=2).draw(output="mpl", fold=-1)

<Image src="../docs/images/tutorials/shors-algorithm/extracted-outputs/13b4841d-a4ac-46bd-b4d0-d111b3017189-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/shors-algorithm/extracted-outputs/13b4841d-a4ac-46bd-b4d0-d111b3017189-0.avif)

Sekarang kita perlu membangun operator eksponensiasi modular. Untuk mendapatkan presisi yang cukup dalam estimasi fase, kita akan menggunakan delapan Qubit untuk pengukuran estimasi. Oleh karena itu, kita perlu membangun $M_b$ dengan $b = a^{2^k} \; (\mathrm{mod} \; N)$ untuk setiap $k = 0, 1, \dots, 7$.

In [7]:
def a2kmodN(a, k, N):
    """Compute a^{2^k} (mod N) by repeated squaring"""
    for _ in range(k):
        a = int(np.mod(a**2, N))
    return a

In [8]:
k_list = range(8)
b_list = [a2kmodN(2, k, 15) for k in k_list]

print(b_list)

[2, 4, 1, 1, 1, 1, 1, 1]


As we can see from the list of $b$ values, in addition to $M_2$ that we previously constructed, we also need to build $M_4$ and $M_1$. Note that $M_1$ acts trivially on the computational basis states, so it is simply the identity operator.

$M_4$ acts on the computational basis states as follows.
$$M_4 \ket{0} = \ket{0} \quad M_4 \ket{5} = \ket{5} \quad M_4 \ket{10} = \ket{10}$$
$$M_4 \ket{1} = \ket{4} \quad M_4 \ket{6} = \ket{9} \quad M_4 \ket{11} = \ket{14}$$
$$M_4 \ket{2} = \ket{8} \quad M_4 \ket{7} = \ket{13} \quad M_4 \ket{12} = \ket{3}$$
$$M_4 \ket{3} = \ket{12} \quad M_4 \ket{8} = \ket{2} \quad M_4 \ket{13} = \ket{7}$$
$$M_4 \ket{4} = \ket{1} \quad M_4 \ket{9} = \ket{6} \quad M_4 \ket{14} = \ket{11}$$

Therefore, this permutation can be constructed with the following swap operation.

In [9]:
def M4mod15():
    """
    M4 (mod 15)
    """
    b = 4
    U = QuantumCircuit(4)

    U.swap(1, 3)
    U.swap(0, 2)

    U = U.to_gate()
    U.name = f"M_{b}"

    return U

In [10]:
# Get the M4 operator
M4 = M4mod15()

# Add it to a circuit and plot
circ = QuantumCircuit(4)
circ.compose(M4, inplace=True)
circ.decompose(reps=2).draw(output="mpl", fold=-1)

<Image src="../docs/images/tutorials/shors-algorithm/extracted-outputs/be041e3d-28b1-453e-983e-184c2366aeb9-0.avif" alt="Output of the previous code cell" />

In [11]:
def controlled_M4mod15():
    """
    Controlled M4 (mod 15)
    """
    b = 4
    U = QuantumCircuit(4)

    U.swap(1, 3)
    U.swap(0, 2)

    U = U.to_gate()
    U.name = f"M_{b}"
    c_U = U.control()

    return c_U

In [12]:
# Get the controlled-M4 operator
controlled_M4 = controlled_M4mod15()

# Add it to a circuit and plot
circ = QuantumCircuit(5)
circ.compose(controlled_M4, inplace=True)
circ.decompose(reps=1).draw(output="mpl", fold=-1)

<Image src="../docs/images/tutorials/shors-algorithm/extracted-outputs/8d943b00-a502-4157-8a0d-13fb1f55e705-0.avif" alt="Output of the previous code cell" />

Gates acting on more than two qubits will be further decomposed into two-qubit gates.

In [13]:
circ.decompose(reps=2).draw(output="mpl", fold=-1)

<Image src="../docs/images/tutorials/shors-algorithm/extracted-outputs/68399eef-5e55-4c95-a8a4-c8efaebd34b9-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/shors-algorithm/extracted-outputs/8d943b00-a502-4157-8a0d-13fb1f55e705-0.avif)

Gate yang bekerja pada lebih dari dua Qubit akan didekomposisi lebih lanjut menjadi Gate dua Qubit.

In [14]:
def mod_mult_gate(b, N):
    """
    Modular multiplication gate from permutation matrix.
    """
    if gcd(b, N) > 1:
        print(f"Error: gcd({b},{N}) > 1")
    else:
        n = floor(log(N - 1, 2)) + 1
        U = np.full((2**n, 2**n), 0)
        for x in range(N):
            U[b * x % N][x] = 1
        for x in range(N, 2**n):
            U[x][x] = 1
        G = UnitaryGate(U)
        G.name = f"M_{b}"
        return G

In [15]:
# Let's build M2 using the permutation matrix definition
M2_other = mod_mult_gate(2, 15)

# Add it to a circuit
circ = QuantumCircuit(4)
circ.compose(M2_other, inplace=True)
circ = circ.decompose()

# Transpile the circuit and get the depth
coupling_map = CouplingMap.from_line(4)
pm = generate_preset_pass_manager(coupling_map=coupling_map)
transpiled_circ = pm.run(circ)

print(f"qubits: {circ.num_qubits}")
print(
    f"2q-depth: {transpiled_circ.depth(lambda x: x.operation.num_qubits==2)}"
)
print(f"2q-size: {transpiled_circ.size(lambda x: x.operation.num_qubits==2)}")
print(f"Operator counts: {transpiled_circ.count_ops()}")
transpiled_circ.decompose().draw(
    output="mpl", fold=-1, style="clifford", idle_wires=False
)

qubits: 4
2q-depth: 94
2q-size: 96
Operator counts: OrderedDict({'cx': 45, 'swap': 32, 'u': 24, 'u1': 7, 'u3': 4, 'unitary': 3, 'circuit-335': 1, 'circuit-338': 1, 'circuit-341': 1, 'circuit-344': 1, 'circuit-347': 1, 'circuit-350': 1, 'circuit-353': 1, 'circuit-356': 1, 'circuit-359': 1, 'circuit-362': 1, 'circuit-365': 1, 'circuit-368': 1, 'circuit-371': 1, 'circuit-374': 1, 'circuit-377': 1, 'circuit-380': 1})


<Image src="../docs/images/tutorials/shors-algorithm/extracted-outputs/c184f6dd-9f80-4487-ac0b-0dd94170b0f0-1.avif" alt="Output of the previous code cell" />

Let's compare these counts with the compiled circuit depth of our manual implementation of the $M_2$ gate.

In [16]:
# Get the M2 operator from our manual construction
M2 = M2mod15()

# Add it to a circuit
circ = QuantumCircuit(4)
circ.compose(M2, inplace=True)
circ = circ.decompose(reps=3)

# Transpile the circuit and get the depth
coupling_map = CouplingMap.from_line(4)
pm = generate_preset_pass_manager(coupling_map=coupling_map)
transpiled_circ = pm.run(circ)

print(f"qubits: {circ.num_qubits}")
print(
    f"2q-depth: {transpiled_circ.depth(lambda x: x.operation.num_qubits==2)}"
)
print(f"2q-size: {transpiled_circ.size(lambda x: x.operation.num_qubits==2)}")
print(f"Operator counts: {transpiled_circ.count_ops()}")
transpiled_circ.draw(
    output="mpl", fold=-1, style="clifford", idle_wires=False
)

qubits: 4
2q-depth: 9
2q-size: 9
Operator counts: OrderedDict({'cx': 9})


<Image src="../docs/images/tutorials/shors-algorithm/extracted-outputs/0235c931-0adb-4972-9fce-32a0341822bf-1.avif" alt="Output of the previous code cell" />

As we can see, the permutation matrix approach resulted in a significantly deep circuit even for a single $M_2$ gate compared to our manual implementation of it. Therefore, we will continue with our previous implementation of the $M_b$ operations.

Now, we are ready to construct the full order finding circuit using our previously defined controlled modular exponentiation operators. In the following code, we also import the [QFT circuit](/docs/api/qiskit/qiskit.circuit.library.QFT) from the Qiskit Circuit library, which uses Hadamard gates on each qubit, a series of controlled-U1 (or Z, depending on the phase) gates, and a layer of swap gates.

In [17]:
# Order finding problem for N = 15 with a = 2
N = 15
a = 2

# Number of qubits
num_target = floor(log(N - 1, 2)) + 1  # for modular exponentiation operators
num_control = 2 * num_target  # for enough precision of estimation

# List of M_b operators in order
k_list = range(num_control)
b_list = [a2kmodN(2, k, 15) for k in k_list]

# Initialize the circuit
control = QuantumRegister(num_control, name="C")
target = QuantumRegister(num_target, name="T")
output = ClassicalRegister(num_control, name="out")
circuit = QuantumCircuit(control, target, output)

# Initialize the target register to the state |1>
circuit.x(num_control)

# Add the Hadamard gates and controlled versions of the
# multiplication gates
for k, qubit in enumerate(control):
    circuit.h(k)
    b = b_list[k]
    if b == 2:
        circuit.compose(
            M2mod15().control(), qubits=[qubit] + list(target), inplace=True
        )
    elif b == 4:
        circuit.compose(
            M4mod15().control(), qubits=[qubit] + list(target), inplace=True
        )
    else:
        continue  # M1 is the identity operator

# Apply the inverse QFT to the control register
circuit.compose(QFT(num_control, inverse=True), qubits=control, inplace=True)

# Measure the control register
circuit.measure(control, output)

circuit.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/shors-algorithm/extracted-outputs/0e854aed-c11b-494c-8c80-adeb8eb0e8fe-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/shors-algorithm/extracted-outputs/c184f6dd-9f80-4487-ac0b-0dd94170b0f0-1.avif)

Mari kita bandingkan jumlah ini dengan kedalaman Circuit yang telah dikompilasi dari implementasi manual Gate $M_2$ kita.

In [ ]:
service = QiskitRuntimeService()
backend = service.backend("ibm_marrakesh")
pm = generate_preset_pass_manager(optimization_level=2, backend=backend)

transpiled_circuit = pm.run(circuit)

print(
    f"2q-depth: {transpiled_circuit.depth(lambda x: x.operation.num_qubits==2)}"
)
print(
    f"2q-size: {transpiled_circuit.size(lambda x: x.operation.num_qubits==2)}"
)
print(f"Operator counts: {transpiled_circuit.count_ops()}")
transpiled_circuit.draw(
    output="mpl", fold=-1, style="clifford", idle_wires=False
)

2q-depth: 187
2q-size: 260
Operator counts: OrderedDict({'sx': 521, 'rz': 354, 'cz': 260, 'measure': 8, 'x': 4})


<Image src="../docs/images/tutorials/shors-algorithm/extracted-outputs/95925dd5-7ba9-4746-b96e-ba50400fa5ac-1.avif" alt="Output of the previous code cell" />

## Step 3: Execute using Qiskit primitives

First, we discuss what we would theoretically obtain if we ran this circuit on an ideal simulator. Below, we have a set of simulation results of the above circuit using 1024 shots. As we can see, we get an approximately uniform distribution over four bitstrings over the control qubits.

In [19]:
# Obtained from the simulator
counts = {"00000000": 264, "01000000": 268, "10000000": 249, "11000000": 243}

In [20]:
plot_histogram(counts)

<Image src="../docs/images/tutorials/shors-algorithm/extracted-outputs/0d6d2702-02e4-47de-8f7e-0b256657ef0f-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/shors-algorithm/extracted-outputs/0e854aed-c11b-494c-8c80-adeb8eb0e8fe-0.avif)

Perhatikan bahwa kita menghilangkan operasi eksponensiasi modular terkontrol dari Qubit kontrol yang tersisa karena $M_1$ adalah operator identitas.
Perhatikan bahwa nantinya dalam tutorial ini, kita akan menjalankan Circuit ini pada Backend `ibm_marrakesh`. Untuk melakukan ini, kita mentranspilasi Circuit sesuai Backend spesifik ini dan melaporkan kedalaman Circuit serta jumlah Gate.

In [21]:
# Rows to be displayed in table
rows = []
# Corresponding phase of each bitstring
measured_phases = []

for output in counts:
    decimal = int(output, 2)  # Convert bitstring to decimal
    phase = decimal / (2**num_control)  # Find corresponding eigenvalue
    measured_phases.append(phase)
    # Add these values to the rows in our table:
    rows.append(
        [
            f"{output}(bin) = {decimal:>3}(dec)",
            f"{decimal}/{2 ** num_control} = {phase:.2f}",
        ]
    )

# Print the rows in a table
headers = ["Register Output", "Phase"]
df = pd.DataFrame(rows, columns=headers)
print(df)

            Register Output           Phase
0  00000000(bin) =   0(dec)    0/256 = 0.00
1  01000000(bin) =  64(dec)   64/256 = 0.25
2  10000000(bin) = 128(dec)  128/256 = 0.50
3  11000000(bin) = 192(dec)  192/256 = 0.75


Recall that the any measured phase corresponds to $\theta = k / r$ where $k$ is sampled uniformly at random from $\{0, 1, \dots, r-1 \}$. Therefore, we can use the continued fractions algorithm to attempt to find $k$ and the order $r$. Python has this functionality built in. We can use the `fractions` module to turn a float into a `Fraction` object, for example:

In [22]:
Fraction(0.666)

Fraction(5998794703657501, 9007199254740992)

![Output of the previous code cell](../docs/images/tutorials/shors-algorithm/extracted-outputs/95925dd5-7ba9-4746-b96e-ba50400fa5ac-1.avif)
## Langkah 3: Eksekusi menggunakan Qiskit primitives
Pertama, kita bahas apa yang secara teoritis akan kita dapatkan jika menjalankan Circuit ini di simulator ideal. Di bawah ini ada sekumpulan hasil simulasi dari Circuit di atas menggunakan 1024 shots. Seperti terlihat, kita mendapatkan distribusi yang kira-kira seragam di empat bitstring pada control qubit.

In [23]:
# Get fraction that most closely resembles 0.666
# with denominator < 15
Fraction(0.666).limit_denominator(15)

Fraction(2, 3)

This is much nicer. The order (r) must be less than N, so we will set the maximum denominator to be `15`:

In [24]:
# Rows to be displayed in a table
rows = []

for phase in measured_phases:
    frac = Fraction(phase).limit_denominator(15)
    rows.append(
        [phase, f"{frac.numerator}/{frac.denominator}", frac.denominator]
    )

# Print the rows in a table
headers = ["Phase", "Fraction", "Guess for r"]
df = pd.DataFrame(rows, columns=headers)
print(df)

   Phase Fraction  Guess for r
0   0.00      0/1            1
1   0.25      1/4            4
2   0.50      1/2            2
3   0.75      3/4            4


![Output of the previous code cell](../docs/images/tutorials/shors-algorithm/extracted-outputs/0d6d2702-02e4-47de-8f7e-0b256657ef0f-0.avif)

Dengan mengukur control qubit, kita mendapatkan estimasi fase delapan-bit dari operator $M_a$. Kita bisa mengonversi representasi biner ini ke desimal untuk menemukan fase yang terukur. Seperti terlihat dari histogram di atas, empat bitstring berbeda diukur, dan masing-masing berkorespondensi dengan nilai fase sebagai berikut.

In [ ]:
# Sampler primitive to obtain the probability distribution
sampler = Sampler(backend)

# Turn on dynamical decoupling with sequence XpXm
sampler.options.dynamical_decoupling.enable = True
sampler.options.dynamical_decoupling.sequence_type = "XpXm"
# Enable gate twirling
sampler.options.twirling.enable_gates = True

pub = transpiled_circuit
job = sampler.run([pub], shots=1024)

In [25]:
result = job.result()[0]
counts = result.data["out"].get_counts()

In [26]:
plot_histogram(counts, figsize=(35, 5))

<Image src="../docs/images/tutorials/shors-algorithm/extracted-outputs/559d7030-1f67-44e8-afa7-6afc7a334677-0.avif" alt="Output of the previous code cell" />

As we can see, we obtained the same bitstrings with highest counts. Since quantum hardware has noise, there is some leakage to other bitstrings, which we can filter out statistically.

In [27]:
# Dictionary of bitstrings and their counts to keep
counts_keep = {}
# Threshold to filter
threshold = np.max(list(counts.values())) / 2

for key, value in counts.items():
    if value > threshold:
        counts_keep[key] = value

print(counts_keep)

{'00000000': 58, '01000000': 41, '11000000': 42, '10000000': 40}


Karena ini menghasilkan pecahan yang mengembalikan hasil secara tepat (dalam kasus ini, `0.6660000...`), hasilnya bisa jadi rumit seperti di atas. Kita bisa menggunakan metode `.limit_denominator()` untuk mendapatkan pecahan yang paling mendekati float kita, dengan penyebut di bawah nilai tertentu:

In [28]:
a = 2
N = 15

FACTOR_FOUND = False
num_attempt = 0

while not FACTOR_FOUND:
    print(f"\nATTEMPT {num_attempt}:")
    # Here, we get the bitstring by iterating over outcomes
    # of a previous hardware run with multiple shots.
    # Instead, we can also perform a single-shot measurement
    # here in the loop.
    bitstring = list(counts_keep.keys())[num_attempt]
    num_attempt += 1
    # Find the phase from measurement
    decimal = int(bitstring, 2)
    phase = decimal / (2**num_control)  # phase = k / r
    print(f"Phase: theta = {phase}")

    # Guess the order from phase
    frac = Fraction(phase).limit_denominator(N)
    r = frac.denominator  # order = r
    print(f"Order of {a} modulo {N} estimated as: r = {r}")

    if phase != 0:
        # Guesses for factors are gcd(a^{r / 2} ± 1, 15)
        if r % 2 == 0:
            x = pow(a, r // 2, N) - 1
            d = gcd(x, N)
            if d > 1:
                FACTOR_FOUND = True
                print(f"*** Non-trivial factor found: {x} ***")


ATTEMPT 0:
Phase: theta = 0.0
Order of 2 modulo 15 estimated as: r = 1

ATTEMPT 1:
Phase: theta = 0.25
Order of 2 modulo 15 estimated as: r = 4
*** Non-trivial factor found: 3 ***


## Discussion

### Related work
In this section, we discuss other milestone work that has demonstrated Shor's algorithm on real hardware.

The seminal work [[3]](#references) from IBM&reg; demonstrated Shor's algorithm for the first time, factoring the number 15 into its prime factors 3 and 5 using a seven-qubit nuclear magnetic resonance (NMR) quantum computer. Another experiment [[4]](#references) factored 15 using photonic qubits. By employing a single qubit recycled multiple times and encoding the work register in higher-dimensional states, the researchers reduced the required number of qubits to one-third of that in the standard protocol, utilizing a two-photon compiled algorithm. A significant paper in the demonstration of Shor's algorithm is [[5]](#references), which uses Kitaev's iterative phase estimation [[8]](#references) technique to reduce the qubit requirement of the algorithm. Authors used seven control qubits and four cache qubits, together with the implementation of modular multipliers. This implementation, however, requires mid-circuit measurements with feed-forward operations and qubit recycling with reset operations. This demonstration was done on an ion-trap quantum computer.

More recent work [[6]](#references) focused on factoring 15, 21, and 35 on IBM Quantum&reg; hardware. Similar to previous work, researchers used a compiled version of the algorithm that employed a semi-classical quantum Fourier transform as proposed by Kitaev to minimize the number of physical qubits and gates. A most recent work [[7]](#references) also performed a proof-of-concept demonstration for factoring the integer 21. This demonstration also involved the use of a compiled version of the quantum phase estimation routine, and built upon the previous demonstration by [[4]](#references). Authors went beyond this work by using a configuration of approximate Toffoli gates with residual phase shifts. The algorithm was implemented on IBM quantum processors using only five qubits, and the presence of entanglement between the control and register qubits was verified successfully.

### Scaling of the algorithm

We note that RSA encryption typically involves key sizes on the order of 2048 to 4096 bits. Attempting to factor a 2048-bit number with Shor's algorithm will result in a quantum circuit with millions of qubits, including the error correction overhead and a circuit depth on the order of a billion, which is beyond the limits of current quantum hardware to execute. Therefore, Shor's algorithm will require either optimized circuit construction methods or robust quantum error correction to be practically viable for breaking modern cryptographic systems. We refer you to [[9]](#references) for a more detailed discussion on resource estimation for Shor's algorithm.

## Challenge

Congratulations for finishing the tutorial! Now is a great time to test your understanding. Could you try to construct the circuit for factoring 21? You can select an $a$ of your own choice. You will need to decide on the bit accuracy of the algorithm to choose the number of qubits, and you will need to design the modular exponentiation operators $M_a$. We encourage you to try this out yourself, and then read about the methodologies shown in Fig. 9 of [[6]](#references) and Fig. 2 of [[7]](#references).

In [ ]:
def M_a_mod21():
    """
    M_a (mod 21)
    """

    # Your code here
    pass

Ini jauh lebih rapi. Orde (r) harus lebih kecil dari N, jadi kita akan menetapkan penyebut maksimum menjadi `15`: